# 🛫 SkyInsight — Enhanced Airline Vision Models  v3.0

Three computer vision models for airline operations:
1. **Cabin Cleanliness Detection** — Clean / Dirty
2. **Crowd Analytics** — low_crowd / high_crowd
3. **Baggage Handling QC** — damaged_luggage / good_luggage

**v3.0 Additions:** Temperature Scaling · MC-Dropout Uncertainty · Grad-CAM++ · AUROC/PR-AUC · Calibration Diagram · Rejection Threshold · JSON Export


In [11]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Run FIRST.  Fixes matplotlib/Python 3.13 compatibility ║
# ╚══════════════════════════════════════════════════════════════════╝
import subprocess, sys, importlib

def _pip(*pkgs):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

# 1. Ensure matplotlib >= 3.8  (required for Python 3.13)
import matplotlib as _mpl
from packaging.version import Version as _V
if _V(_mpl.__version__) < _V("3.8.0"):
    print("⬆️  Upgrading matplotlib …"); _pip("matplotlib>=3.8.0")

# 2. Ensure matplotlib_inline >= 0.1.7  (fixes backend_agg import error)
try:
    import matplotlib_inline as _mli
    if _V(_mli.__version__) < _V("0.1.7"):
        print("⬆️  Upgrading matplotlib_inline …"); _pip("matplotlib_inline>=0.1.7")
except ImportError:
    _pip("matplotlib_inline>=0.1.7")

# 3. Ensure scipy (needed for temperature scaling)
try:
    import scipy
except ImportError:
    print("📦 Installing scipy …"); _pip("scipy")

# 4. Set Agg backend BEFORE any other matplotlib import
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
plt.rcParams.update({"figure.dpi":120,"savefig.dpi":120,"figure.facecolor":"white"})

# 5. Patch plt.show() to render inline inside VS Code / Jupyter with Agg
import io
from IPython.display import display, Image as _IPyImage

_orig_show = plt.show
def _inline_show(*a, **kw):
    fig = plt.gcf()
    if fig.get_axes():
        buf = io.BytesIO()
        fig.savefig(buf, format="png", bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        buf.seek(0); display(_IPyImage(data=buf.read())); buf.close()
    plt.close("all")
plt.show = _inline_show

print(f"✅ Environment ready  |  matplotlib {matplotlib.__version__}  |  backend: Agg")


✅ Environment ready  |  matplotlib 3.10.9  |  backend: Agg


In [12]:
import os, sys, shutil, random, math, io, json as _json
import numpy as np
import pandas as pd
import scipy.special
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from datetime import datetime
from collections import defaultdict
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, average_precision_score
)
from sklearn.calibration import calibration_curve
import warnings; warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# Mixed precision
try:
    mixed_precision.set_global_policy('mixed_float16')
    print("✅ Mixed precision: float16")
except Exception:
    print("ℹ️  Mixed precision unavailable — float32")

print(f"TensorFlow : {tf.__version__}")
print(f"GPUs found : {tf.config.list_physical_devices('GPU')}")


✅ Mixed precision: float16
TensorFlow : 2.20.0
GPUs found : []


In [13]:
# ── Paths ────────────────────────────────────────────────────────────────────
# Auto-detect repo root that contains 08_CNN/data
from pathlib import Path
_root_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
DL_ROOT = None
for root in _root_candidates:
    if (root / "08_CNN" / "data").exists():
        DL_ROOT = str(root)
        break
if DL_ROOT is None:
    DL_ROOT = str(Path.cwd())

MODELS_DIR = os.path.join(DL_ROOT, "08_CNN", "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# ── Global hyper-parameters ───────────────────────────────────────────────────
BATCH_SIZE          = 32
LABEL_SMOOTHING     = 0.05
MIXUP_ALPHA         = 0.2
TTA_STEPS           = 5
WARMUP_RATIO        = 0.1
REJECTION_THRESHOLD = 0.65   # below this → prediction flagged UNCERTAIN

# ── Per-dataset configs ───────────────────────────────────────────────────────
DATASETS = {
    'cabin_cleanliness': {
        'path'            : os.path.join(DL_ROOT, "08_CNN/data/cabin_cleanliness"),
        'classes'         : ['Clean', 'Dirty'],
        'num_classes'     : 2,
        'img_size'        : 224,
        'backbone'        : 'EfficientNetV2S',
        'epochs_head'     : 20,
        'epochs_fine_tune': 15,
        'lr_head'         : 1e-3,
        'lr_fine'         : 5e-6,
        'weight_decay'    : 1e-4,
        'unfreeze_ratio'  : 0.40,
        'dropout_1'       : 0.45,
        'dropout_2'       : 0.30,
        'use_focal'       : True,
        'focal_gamma'     : 2.5,
        'aug': {'rotation_range':15,'width_shift_range':0.15,'height_shift_range':0.15,
                'shear_range':0.10,'zoom_range':0.20,'horizontal_flip':True,
                'brightness_range':[0.8,1.2],'channel_shift_range':15.0},
    },
    'crowd': {
        'path'            : os.path.join(DL_ROOT, "08_CNN/data/crowd"),
        'classes'         : ['low_crowd','high_crowd'],
        'num_classes'     : 2,
        'img_size'        : 224,
        'backbone'        : 'EfficientNetV2S',
        'epochs_head'     : 22,
        'epochs_fine_tune': 10,
        'lr_head'         : 8e-4,
        'lr_fine'         : 5e-6,
        'weight_decay'    : 1e-4,
        'unfreeze_ratio'  : 0.25,
        'dropout_1'       : 0.40,
        'dropout_2'       : 0.25,
        'use_focal'       : True,
        'focal_gamma'     : 2.0,
        'aug': {'rotation_range':5,'width_shift_range':0.08,'height_shift_range':0.08,
                'shear_range':0.04,'zoom_range':0.10,'horizontal_flip':False,
                'brightness_range':[0.85,1.15],'channel_shift_range':10.0},
    },
    'luggage': {
        'path'            : os.path.join(DL_ROOT, "08_CNN/data/luggage"),
        'classes'         : ['damaged_luggage','good_luggage'],
        'num_classes'     : 2,
        'img_size'        : 224,
        'backbone'        : 'DenseNet121',
        'epochs_head'     : 16,
        'epochs_fine_tune': 12,
        'lr_head'         : 8e-4,
        'lr_fine'         : 8e-6,
        'weight_decay'    : 1e-4,
        'unfreeze_ratio'  : 0.30,
        'dropout_1'       : 0.40,
        'dropout_2'       : 0.30,
        'use_focal'       : False,
        'focal_gamma'     : 2.0,
        'aug': {'rotation_range':10,'width_shift_range':0.12,'height_shift_range':0.12,
                'shear_range':0.08,'zoom_range':0.15,'horizontal_flip':True,
                'brightness_range':[0.8,1.2],'channel_shift_range':12.0},
    }
}

print(f"✅ Configuration loaded (DL_ROOT={DL_ROOT})")
for name, cfg in DATASETS.items():
    print(f"  {name}: backbone={cfg['backbone']}, focal={cfg['use_focal']}")


✅ Configuration loaded (DL_ROOT=c:\Users\achre\Downloads\Esprit\DL\SkyInsight)
  cabin_cleanliness: backbone=EfficientNetV2S, focal=True
  crowd: backbone=EfficientNetV2S, focal=True
  luggage: backbone=DenseNet121, focal=False


In [14]:
# ── Backbone factory ─────────────────────────────────────────────────────────
def _get_preprocess_fn(backbone_name):
    mapping = {
        'EfficientNetB0' : keras.applications.efficientnet.preprocess_input,
        'EfficientNetV2S': keras.applications.efficientnet_v2.preprocess_input,
        'ResNet50V2'     : keras.applications.resnet_v2.preprocess_input,
        'DenseNet121'    : keras.applications.densenet.preprocess_input,
        'ConvNeXtTiny'   : keras.applications.convnext.preprocess_input,
    }
    if backbone_name not in mapping:
        raise ValueError(f"Unsupported backbone: {backbone_name}")
    return mapping[backbone_name]

def _get_backbone(backbone_name, input_shape):
    kwargs = dict(include_top=False, weights='imagenet', input_shape=input_shape)
    constructors = {
        'EfficientNetB0' : keras.applications.EfficientNetB0,
        'EfficientNetV2S': keras.applications.EfficientNetV2S,
        'ResNet50V2'     : keras.applications.ResNet50V2,
        'DenseNet121'    : keras.applications.DenseNet121,
        'ConvNeXtTiny'   : keras.applications.ConvNeXtTiny,
    }
    if backbone_name not in constructors:
        raise ValueError(f"Unsupported backbone: {backbone_name}")
    base = constructors[backbone_name](**kwargs)
    return base, _get_preprocess_fn(backbone_name)

# ── GeM pooling layer ─────────────────────────────────────────────────────────
class GeneralizedMeanPooling(layers.Layer):
    """GeM Pooling (p=3) — better than GAP for fine-grained tasks."""
    def __init__(self, p=3.0, **kwargs):
        super().__init__(**kwargs); self.p = p
    def call(self, x):
        x = tf.clip_by_value(x, 1e-6, tf.reduce_max(x))
        return tf.pow(tf.reduce_mean(tf.pow(x, self.p), axis=[1,2]), 1.0/self.p)
    def get_config(self):
        cfg = super().get_config(); cfg['p'] = self.p; return cfg

# ── Model builder ─────────────────────────────────────────────────────────────
def create_model(dataset_name):
    """Transfer-learning model with GeM pooling + residual head."""
    cfg         = DATASETS[dataset_name]
    img_size    = cfg['img_size']
    input_shape = (img_size, img_size, 3)
    base_model, preprocess_fn = _get_backbone(cfg['backbone'], input_shape)
    base_model.trainable = False   # frozen during head training

    inputs = keras.Input(shape=input_shape)
    x = layers.Lambda(preprocess_fn, name='preprocess')(inputs)
    x = base_model(x, training=False)
    x = GeneralizedMeanPooling(p=3.0, name='gem_pool')(x)
    x = layers.BatchNormalization(name='head_bn1')(x)
    shortcut = layers.Dense(256, name='shortcut')(x)
    x = layers.Dense(256, activation='swish',
                     kernel_regularizer=keras.regularizers.l2(cfg['weight_decay']),
                     name='fc1')(x)
    x = layers.Dropout(cfg['dropout_1'], name='drop1')(x)
    x = layers.Add(name='res_add')([x, shortcut])
    x = layers.BatchNormalization(name='head_bn2')(x)
    x = layers.Dense(128, activation='swish',
                     kernel_regularizer=keras.regularizers.l2(cfg['weight_decay']),
                     name='fc2')(x)
    x = layers.Dropout(cfg['dropout_2'], name='drop2')(x)
    x = layers.Dense(cfg['num_classes'], name='logits')(x)
    outputs = layers.Activation('softmax', dtype='float32', name='predictions')(x)
    return keras.Model(inputs, outputs, name=f"{dataset_name}_model"), base_model

print("✅ Backbone factory + GeM model builder defined")


✅ Backbone factory + GeM model builder defined


In [15]:
def load_data(dataset_name, batch_size=BATCH_SIZE):
    cfg      = DATASETS[dataset_name]
    img_size = cfg['img_size']
    aug      = cfg['aug']
    train_datagen = ImageDataGenerator(
        rescale             = 1.0/255.0,
        rotation_range      = aug['rotation_range'],
        width_shift_range   = aug['width_shift_range'],
        height_shift_range  = aug['height_shift_range'],
        shear_range         = aug['shear_range'],
        zoom_range          = aug['zoom_range'],
        horizontal_flip     = aug['horizontal_flip'],
        brightness_range    = aug.get('brightness_range'),
        channel_shift_range = aug.get('channel_shift_range', 0.0),
        fill_mode           = 'reflect'
    )
    val_test_datagen = ImageDataGenerator(rescale=1.0/255.0)
    def make_gen(directory, datagen, shuffle):
        return datagen.flow_from_directory(
            directory, target_size=(img_size,img_size),
            batch_size=batch_size, class_mode='categorical',
            shuffle=shuffle, seed=SEED
        )
    base = cfg['path']
    return (make_gen(os.path.join(base,'train'), train_datagen, True),
            make_gen(os.path.join(base,'val'),   val_test_datagen, False),
            make_gen(os.path.join(base,'test'),  val_test_datagen, False))

def mixup_generator(generator, alpha=MIXUP_ALPHA):
    """Wrap a generator with Mixup augmentation."""
    while True:
        X1,y1 = next(generator); X2,y2 = next(generator)
        n = min(len(X1),len(X2))
        X1,y1,X2,y2 = X1[:n],y1[:n],X2[:n],y2[:n]
        lam = np.random.beta(alpha,alpha,size=(n,1,1,1)).astype(np.float32)
        yield lam*X1+(1-lam)*X2, lam.reshape(n,1)*y1+(1-lam.reshape(n,1))*y2

print("✅ Data loaders + Mixup generator defined")


✅ Data loaders + Mixup generator defined


In [16]:
def categorical_focal_loss(alpha_vec, gamma=2.0):
    alpha = tf.constant(alpha_vec, dtype=tf.float32)
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0-1e-7)
        weight = tf.pow(1.0-y_pred, gamma)*alpha
        return tf.reduce_mean(tf.reduce_sum(weight*(-y_true*tf.math.log(y_pred)), axis=-1))
    return loss_fn

def cosine_annealing_schedule(lr_max, total_epochs, warmup_epochs=0):
    def schedule(epoch):
        if epoch < warmup_epochs:
            return lr_max*(epoch+1)/max(warmup_epochs,1)
        progress = (epoch-warmup_epochs)/max(total_epochs-warmup_epochs,1)
        return lr_max*0.5*(1+math.cos(math.pi*progress))
    return schedule

print("✅ Focal loss + cosine LR schedule defined")


✅ Focal loss + cosine LR schedule defined


In [17]:
def evaluate_with_tta(model, test_gen, n_passes=TTA_STEPS):
    """Average predictions over multiple augmented forward passes."""
    aug_gen = ImageDataGenerator(
        rescale=1.0, horizontal_flip=True, zoom_range=0.05,
        rotation_range=5, brightness_range=[0.9,1.1]
    )
    test_gen.reset()
    images, labels = [], []
    for _ in range(math.ceil(test_gen.samples/test_gen.batch_size)):
        X,y = next(test_gen); images.append(X); labels.append(y)
    images = np.concatenate(images)[:test_gen.samples]
    labels = np.concatenate(labels)[:test_gen.samples]

    preds = model.predict(images, verbose=0, batch_size=test_gen.batch_size)
    for _ in range(n_passes-1):
        aug   = np.array([aug_gen.random_transform(img) for img in images])
        preds += model.predict(aug, verbose=0, batch_size=test_gen.batch_size)
    preds /= n_passes
    y_true = np.argmax(labels, axis=1)
    y_pred = np.argmax(preds,  axis=1)
    return {
        'accuracy'          : float(accuracy_score(y_true, y_pred)),
        'precision_macro'   : float(precision_score(y_true,y_pred,average='macro',  zero_division=0)),
        'recall_macro'      : float(recall_score   (y_true,y_pred,average='macro',  zero_division=0)),
        'f1_macro'          : float(f1_score        (y_true,y_pred,average='macro',  zero_division=0)),
        'precision_weighted': float(precision_score(y_true,y_pred,average='weighted',zero_division=0)),
        'f1_weighted'       : float(f1_score        (y_true,y_pred,average='weighted',zero_division=0)),
        'y_true': y_true, 'y_pred': y_pred, 'y_prob': preds
    }

print("✅ TTA evaluation defined")


✅ TTA evaluation defined


In [18]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = keras.Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, preds = grad_model(img_array, training=False)
        if pred_index is None: pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads   = tape.gradient(class_channel, conv_outputs)
    pooled  = tf.reduce_mean(grads, axis=(0,1,2))
    heatmap = tf.squeeze(conv_outputs[0] @ pooled[..., tf.newaxis])
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap)+1e-8)
    return heatmap.numpy()

def _find_last_conv(model):
    for layer in reversed(model.layers):
        if isinstance(layer,(layers.Conv2D,layers.DepthwiseConv2D)):
            return layer.name
        if hasattr(layer,'layers'):
            for sub in reversed(layer.layers):
                if isinstance(sub,(layers.Conv2D,layers.DepthwiseConv2D)):
                    return sub.name
    return None

def _overlay_heatmap(arr, heatmap, img_size):
    hm_rsz = np.array(
        keras.preprocessing.image.array_to_img(heatmap[...,np.newaxis])
        .resize((img_size, img_size))
    )/255.0
    return 0.6*arr + 0.4*cm.jet(hm_rsz)[:,:,:3]

def visualize_gradcam(model, dataset_name, class_names, n_images=4):
    cfg      = DATASETS[dataset_name]
    test_dir = os.path.join(cfg['path'],'test')
    img_size = cfg['img_size']
    last_conv = _find_last_conv(model)
    if last_conv is None:
        print("⚠️  No conv layer found — skipping Grad-CAM."); return

    paths, labs = [], []
    for cls in os.listdir(test_dir):
        d = os.path.join(test_dir,cls)
        if not os.path.isdir(d): continue
        imgs = [f for f in os.listdir(d) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        for f in random.sample(imgs, min(n_images//max(len(class_names),1)+1,len(imgs))):
            paths.append(os.path.join(d,f)); labs.append(cls)
    paths,labs = paths[:n_images],labs[:n_images]
    n = len(paths)

    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
    if n==1: axes = axes.reshape(2,1)
    fig.suptitle(f"{dataset_name.upper()} — Grad-CAM", fontsize=14, fontweight='bold')

    for i,(path,label) in enumerate(zip(paths,labs)):
        img = keras.preprocessing.image.load_img(path, target_size=(img_size,img_size))
        arr = keras.preprocessing.image.img_to_array(img)/255.0
        inp = np.expand_dims(arr,0)
        try:
            heatmap = make_gradcam_heatmap(inp, model, last_conv)
            overlay = _overlay_heatmap(arr, heatmap, img_size)
        except Exception as e:
            overlay = arr; print(f"  Grad-CAM error: {e}")
        pred    = model.predict(inp, verbose=0)[0]
        p_cls   = class_names[np.argmax(pred)]; conf = np.max(pred)
        axes[0,i].imshow(arr); axes[0,i].axis('off')
        axes[0,i].set_title(f"True: {label}", fontsize=9)
        axes[1,i].imshow(overlay); axes[1,i].axis('off')
        axes[1,i].set_title(f"Pred: {p_cls} ({conf:.2f})", fontsize=9,
                            color='green' if p_cls==label else 'red')
    plt.tight_layout()
    plt.savefig(os.path.join(MODELS_DIR,f"{dataset_name}_gradcam.png"),
                dpi=100, bbox_inches='tight')
    plt.show()
    print(f"✅ Grad-CAM saved: {dataset_name}_gradcam.png")

print("✅ Grad-CAM visualizer defined")


✅ Grad-CAM visualizer defined


In [19]:
def train_model(dataset_name, batch_size=BATCH_SIZE, use_mixup=True):
    cfg = DATASETS[dataset_name]
    print("\n"+"═"*72)
    print(f"  🚀  TRAINING: {dataset_name.upper()}  |  Backbone: {cfg['backbone']}")
    print("═"*72)

    train_gen, val_gen, test_gen = load_data(dataset_name, batch_size)
    class_names = sorted(train_gen.class_indices.keys())
    print(f"🏷️  Classes : {class_names}")
    print(f"📊 Samples : train={train_gen.samples} | val={val_gen.samples} | test={test_gen.samples}")

    # Class weights
    cls_counts  = np.bincount(train_gen.classes)
    n_classes   = len(cls_counts); n_samples = np.sum(cls_counts)
    cw = {i: float(n_samples/(n_classes*max(1,cls_counts[i]))) for i in range(n_classes)}
    cw = {k:(0.85*v+0.15) for k,v in cw.items()}
    cw_sum    = sum(cw.values())
    alpha_vec = np.array([cw[i]/cw_sum for i in range(n_classes)], dtype=np.float32)
    print(f"⚖️  Class weights: { {k:round(v,3) for k,v in cw.items()} }")

    model, base_model = create_model(dataset_name)

    if cfg['use_focal']:
        train_loss = categorical_focal_loss(alpha_vec, gamma=cfg.get('focal_gamma',2.0))
        print(f"🎯 Loss: Focal (γ={cfg.get('focal_gamma',2.0)})")
    else:
        train_loss = keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
        print(f"🎯 Loss: LabelSmoothing CE (ε={LABEL_SMOOTHING})")

    warmup_epochs = max(2, int(cfg['epochs_head']*WARMUP_RATIO))
    head_schedule = cosine_annealing_schedule(cfg['lr_head'],cfg['epochs_head'],warmup_epochs)
    timestamp     = datetime.now().strftime("%Y%m%d_%H%M%S")
    ckpt_path     = os.path.join(MODELS_DIR, f"{dataset_name}_best_{timestamp}.keras")

    callbacks_head = [
        keras.callbacks.LearningRateScheduler(head_schedule, verbose=0),
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=7,
            restore_best_weights=True, verbose=1, mode='max'),
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor='val_accuracy',
            save_best_only=True, mode='max', verbose=1)
    ]

    # Stage 1: head training
    print(f"\n📚 Stage 1/2: Head training ({cfg['epochs_head']} epochs, {warmup_epochs}-ep warmup)")
    base_model.trainable = False
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=cfg['lr_head'],weight_decay=cfg['weight_decay']),
        loss=train_loss, metrics=['accuracy']
    )
    steps_per_epoch = math.ceil(train_gen.samples/batch_size)
    if use_mixup:
        train_input = mixup_generator(train_gen, alpha=MIXUP_ALPHA)
        print(f"🔀 Mixup enabled (α={MIXUP_ALPHA})")
    else:
        train_input = train_gen

    history_head = model.fit(
        train_input, steps_per_epoch=steps_per_epoch, validation_data=val_gen,
        epochs=cfg['epochs_head'], callbacks=callbacks_head,
        class_weight=cw if not use_mixup else None, verbose=1
    )

    # Stage 2: fine-tune
    history_fine = None
    if cfg['epochs_fine_tune'] > 0:
        print(f"\n🛠️  Stage 2/2: Fine-tuning top {int(cfg['unfreeze_ratio']*100)}% "
              f"of backbone ({cfg['epochs_fine_tune']} epochs)")
        base_model.trainable = True
        fine_at = int(len(base_model.layers)*(1.0-cfg['unfreeze_ratio']))
        for layer in base_model.layers[:fine_at]:
            layer.trainable = False
        fine_schedule = cosine_annealing_schedule(cfg['lr_fine'],cfg['epochs_fine_tune'],1)
        last_ep = history_head.epoch[-1]+1
        callbacks_fine = [
            keras.callbacks.LearningRateScheduler(
                lambda ep: fine_schedule(ep-last_ep), verbose=0),
            keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5,
                restore_best_weights=True, verbose=1, mode='max'),
            keras.callbacks.ModelCheckpoint(ckpt_path, monitor='val_accuracy',
                save_best_only=True, mode='max', verbose=1)
        ]
        model.compile(
            optimizer=keras.optimizers.AdamW(learning_rate=cfg['lr_fine'],
                                             weight_decay=cfg['weight_decay']*0.1),
            loss=train_loss, metrics=['accuracy']
        )
        train_gen.reset()
        train_input_fine = mixup_generator(train_gen,alpha=MIXUP_ALPHA*0.5) if use_mixup else train_gen
        history_fine = model.fit(
            train_input_fine, steps_per_epoch=steps_per_epoch, validation_data=val_gen,
            initial_epoch=last_ep, epochs=last_ep+cfg['epochs_fine_tune'],
            callbacks=callbacks_fine, class_weight=cw if not use_mixup else None, verbose=1
        )

    # Merge histories
    history = dict(history_head.history)
    if history_fine:
        for k,v in history_fine.history.items():
            history[k] = history.get(k,[])+v

    # TTA Evaluation
    print(f"\n🔬 Evaluating with TTA ({TTA_STEPS} passes)…")
    test_gen.reset()
    em = evaluate_with_tta(model, test_gen, n_passes=TTA_STEPS)
    print("\n✅ Test Results (TTA, macro):")
    print(f"   Accuracy  : {em['accuracy']:.4f}")
    print(f"   Precision : {em['precision_macro']:.4f}")
    print(f"   Recall    : {em['recall_macro']:.4f}")
    print(f"   F1 (macro): {em['f1_macro']:.4f}")
    print(f"   F1 (wtd)  : {em['f1_weighted']:.4f}")
    print("\n📋 Per-class Report:")
    print(classification_report(em['y_true'],em['y_pred'],target_names=class_names,zero_division=0))

    results = {
        'dataset':dataset_name,'backbone':cfg['backbone'],
        'test_accuracy':em['accuracy'],'test_precision':em['precision_macro'],
        'test_recall':em['recall_macro'],'test_f1':em['f1_macro'],
        'test_f1_weighted':em['f1_weighted'],
        'y_true':em['y_true'],'y_pred':em['y_pred'],'y_prob':em['y_prob'],
        'history':history,'class_names':class_names
    }

    # Plots: Loss / Accuracy / Confusion Matrix
    fig, axes = plt.subplots(1,3,figsize=(18,4))
    fig.suptitle(f"{dataset_name.upper()} — {cfg['backbone']}", fontsize=14, fontweight='bold')
    axes[0].plot(history['loss'],     label='Train',lw=2)
    axes[0].plot(history['val_loss'], label='Val',  lw=2)
    axes[0].set_title('Loss');  axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(True,alpha=0.3)
    axes[1].plot(history['accuracy'],     label='Train',lw=2)
    axes[1].plot(history['val_accuracy'], label='Val',  lw=2)
    axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(True,alpha=0.3)
    cm_vals = confusion_matrix(em['y_true'],em['y_pred'])
    disp    = ConfusionMatrixDisplay(confusion_matrix=cm_vals, display_labels=class_names)
    disp.plot(ax=axes[2], colorbar=False, cmap='Blues', values_format='d')
    axes[2].set_title('Confusion Matrix (TTA)')
    plt.tight_layout()
    plot_path = os.path.join(MODELS_DIR,f"{dataset_name}_history_{timestamp}.png")
    plt.savefig(plot_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f"📈 Training plot saved: {plot_path}")

    # Load best checkpoint
    if os.path.exists(ckpt_path):
        try:
            model = keras.models.load_model(
                ckpt_path,
                custom_objects={'GeneralizedMeanPooling':GeneralizedMeanPooling,
                                'loss_fn':train_loss},
                compile=False
            )
            print(f"💾 Best checkpoint loaded: {ckpt_path}")
        except Exception as e:
            print(f"⚠️  Could not reload checkpoint ({e}) — using in-memory model")

    return model, results, class_names

print("✅ train_model() defined")


✅ train_model() defined


## 1️⃣  Cabin Cleanliness Detection

In [ ]:
cabin_model, cabin_results, cabin_classes = train_model('cabin_cleanliness')


════════════════════════════════════════════════════════════════════════
  🚀  TRAINING: CABIN_CLEANLINESS  |  Backbone: EfficientNetV2S
════════════════════════════════════════════════════════════════════════
Found 429 images belonging to 2 classes.
Found 92 images belonging to 2 classes.
Found 94 images belonging to 2 classes.
🏷️  Classes : ['Clean', 'Dirty']


📊 Samples : train=429 | val=92 | test=94
⚖️  Class weights: {0: 0.75, 1: 1.609}

🎯 Loss: Focal (γ=2.5)

📚 Stage 1/2: Head training (20 epochs, 2-ep warmup)
🔀 Mixup enabled (α=0.2)
Epoch 1/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.5547 - loss: 0.2648
Epoch 1: val_accuracy improved from None to 0.29348, saving model to c:\Users\achre\Downloads\Esprit\DL\SkyInsight\08_CNN\models\cabin_cleanliness_best_20260508_121257.keras

Epoch 1: finished saving model to c:\Users\achre\Downloads\Esprit\DL\SkyInsight\08_CNN\models\cabin_cleanliness_best_20260508_121257.keras
14/14 ━━━━━━━━━━━━━━━━━━━━ 119s 7s/step - accuracy: 0.5463 - loss: 0.2657 - val_accuracy: 0.2935 - val_loss: 0.4021 - learning_rate: 5.0000e-04
Epoch 2/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.4533 - loss: 0.2412
Epoch 2: val_accuracy improved from 0.29348 to 0.70652, saving model to c:\Users\achre\Downloads\Esprit\DL\SkyInsight\08_CNN\models\cabin_cleanliness_best_20260508_121257.keras

Epoch 2: finished

In [ ]:
visualize_gradcam(cabin_model, 'cabin_cleanliness', cabin_classes, n_images=4)

## 2️⃣  Crowd Analytics

In [ ]:
crowd_model, crowd_results, crowd_classes = train_model('crowd')

In [ ]:
visualize_gradcam(crowd_model, 'crowd', crowd_classes, n_images=4)

## 3️⃣  Baggage Handling QC

In [ ]:
luggage_model, luggage_results, luggage_classes = train_model('luggage')

In [ ]:
visualize_gradcam(luggage_model, 'luggage', luggage_classes, n_images=4)

## 📊 Final Results & Comparison

In [ ]:
rows = []
for r in [cabin_results, crowd_results, luggage_results]:
    rows.append({
        'Model'            : r['dataset'].upper(),
        'Backbone'         : r['backbone'],
        'Accuracy'         : f"{r['test_accuracy']:.4f}",
        'Precision (macro)': f"{r['test_precision']:.4f}",
        'Recall (macro)'   : f"{r['test_recall']:.4f}",
        'F1 (macro)'       : f"{r['test_f1']:.4f}",
        'F1 (weighted)'    : f"{r['test_f1_weighted']:.4f}",
    })

results_df = pd.DataFrame(rows)
print("\n"+"═"*90)
print("  🏆  FINAL RESULTS  (with TTA)")
print("═"*90)
print(results_df.to_string(index=False))
print("═"*90)

metrics = ['Accuracy','F1 (macro)','F1 (weighted)']
x = np.arange(len(rows)); width = 0.25
fig, ax = plt.subplots(figsize=(10,5))
for i,m in enumerate(metrics):
    vals = [float(r[m]) for r in rows]
    bars = ax.bar(x+i*width, vals, width, label=m, alpha=0.85)
    ax.bar_label(bars, fmt='%.3f', fontsize=8, padding=2)
ax.set_xticks(x+width); ax.set_xticklabels([r['Model'] for r in rows])
ax.set_ylim(0,1.12); ax.set_ylabel('Score')
ax.set_title('Model Comparison — Test Metrics (with TTA)')
ax.legend(); ax.grid(axis='y',alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR,'comparison_chart.png'),dpi=100)
plt.show()

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(MODELS_DIR,f"training_results_{ts}.csv")
results_df.to_csv(csv_path,index=False)
print(f"✅ Results CSV saved: {csv_path}")


## 🔮 Advanced Prediction Pipeline — v3.0

| Enhancement | Purpose |
|---|---|
| **Temperature Scaling** | Calibrate overconfident softmax probabilities |
| **MC-Dropout (×15)** | Epistemic uncertainty with per-class error bands |
| **Ensemble TTA+MC** | Weighted combination for maximum robustness |
| **Rejection Threshold** | Abstain when confidence < 0.65 |
| **Grad-CAM++** | Sharper localization than standard Grad-CAM |
| **AUROC / PR-AUC** | Richer binary evaluation per class |
| **Calibration Curve** | Reliability diagram before/after scaling |
| **JSON Export** | BI-ready per-image prediction records |


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  1. TEMPERATURE SCALING
# ────────────────────────────────────────────────────────────────────────────
def find_temperature(model, val_gen, T_range=np.linspace(0.5,5.0,91)):
    """Grid-search temperature T that minimises NLL on the validation set."""
    val_gen.reset()
    images, labels = [], []
    for _ in range(math.ceil(val_gen.samples/val_gen.batch_size)):
        X,y = next(val_gen); images.append(X); labels.append(y)
    images = np.concatenate(images)[:val_gen.samples]
    labels = np.concatenate(labels)[:val_gen.samples]

    # Try to extract logits layer; fall back to using log-softmax
    try:
        logit_model = keras.Model(inputs=model.input,
                                  outputs=model.get_layer('logits').output)
        logits = logit_model.predict(images, verbose=0, batch_size=32)
        use_logits = True
    except Exception:
        raw_preds  = model.predict(images, verbose=0, batch_size=32)
        logits     = np.log(raw_preds + 1e-9)
        use_logits = False

    best_T, best_nll = 1.0, np.inf
    for T in T_range:
        scaled = scipy.special.softmax(logits/T, axis=-1)
        nll    = -np.mean(np.sum(labels*np.log(scaled+1e-9), axis=-1))
        if nll < best_nll:
            best_nll, best_T = nll, float(T)
    print(f"  📐 Best temperature T={best_T:.2f}  (NLL={best_nll:.4f})")
    return best_T

def predict_calibrated(model, images, temperature):
    """Temperature-scaled predictions."""
    try:
        logit_model = keras.Model(inputs=model.input,
                                  outputs=model.get_layer('logits').output)
        logits = logit_model.predict(images, verbose=0, batch_size=32)
    except Exception:
        raw    = model.predict(images, verbose=0, batch_size=32)
        logits = np.log(raw+1e-9)
    return scipy.special.softmax(logits/temperature, axis=-1)

# ────────────────────────────────────────────────────────────────────────────
#  2. MONTE-CARLO DROPOUT
# ────────────────────────────────────────────────────────────────────────────
def mc_dropout_predict(model, images, n_passes=15):
    """Stochastic forward passes (dropout ON) → mean + std per class."""
    all_preds = [model(images, training=True).numpy() for _ in range(n_passes)]
    stacked   = np.stack(all_preds, axis=0)   # (P, N, C)
    return np.mean(stacked, axis=0), np.std(stacked, axis=0)  # (N,C),(N,C)

# ────────────────────────────────────────────────────────────────────────────
#  3. REJECTION
# ────────────────────────────────────────────────────────────────────────────
def classify_with_rejection(preds, class_names, threshold=REJECTION_THRESHOLD):
    pred_idx = np.argmax(preds, axis=-1)
    max_conf = np.max(preds,   axis=-1)
    return [(class_names[i] if c>=threshold else "UNCERTAIN", c)
            for i,c in zip(pred_idx, max_conf)]

# ────────────────────────────────────────────────────────────────────────────
#  4. ADVANCED VISUALISATION  (image + prob bars + MC bands)
# ────────────────────────────────────────────────────────────────────────────
def visualize_predictions_advanced(model, dataset_name, class_names,
                                   num_samples=6, temperature=1.0,
                                   mc_passes=15, tta_steps=TTA_STEPS):
    cfg      = DATASETS[dataset_name]
    test_dir = os.path.join(cfg['path'],'test')
    img_size = cfg['img_size']
    aug_gen  = ImageDataGenerator(rescale=1.0, horizontal_flip=True,
                                  zoom_range=0.08, rotation_range=8,
                                  brightness_range=[0.88,1.12])

    images_list, labels_list = [], []
    per_class = max(1, num_samples//max(len(class_names),1))
    for cls in class_names:
        d = os.path.join(test_dir,cls)
        if not os.path.isdir(d): continue
        files = [f for f in os.listdir(d) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        for f in random.sample(files, min(per_class,len(files))):
            img = keras.preprocessing.image.load_img(os.path.join(d,f),
                      target_size=(img_size,img_size))
            arr = keras.preprocessing.image.img_to_array(img)/255.0
            images_list.append(arr); labels_list.append(cls)
    images = np.array(images_list[:num_samples]); labels = labels_list[:num_samples]
    n = len(images)
    if n == 0:
        print("⚠️  No test images found — skipping visualisation."); return

    # TTA (calibrated)
    tta_preds = predict_calibrated(model, images, temperature)
    for _ in range(tta_steps-1):
        aug = np.array([aug_gen.random_transform(im) for im in images])
        tta_preds += predict_calibrated(model, aug, temperature)
    tta_preds /= tta_steps

    # MC-Dropout
    mc_mean, mc_std = mc_dropout_predict(model, images, n_passes=mc_passes)

    final_preds = 0.60*tta_preds + 0.40*mc_mean
    entropy     = -np.sum(final_preds*np.log(final_preds+1e-9), axis=-1)
    decisions   = classify_with_rejection(final_preds, class_names)

    palette    = ['#2196F3','#FF5722','#4CAF50','#9C27B0','#FF9800','#00BCD4']
    bar_colors = palette[:len(class_names)]

    fig, axes = plt.subplots(2, n, figsize=(4.5*n, 9))
    if n==1: axes = axes.reshape(2,1)
    fig.suptitle(
        f"{dataset_name.upper()} — Ensemble Predictions  "
        f"(TTA×{tta_steps} + MC×{mc_passes} + T={temperature:.2f})",
        fontsize=13, fontweight='bold', y=1.01)

    for idx in range(n):
        pred_lbl, conf = decisions[idx]
        ent   = entropy[idx]
        probs = final_preds[idx]; stds = mc_std[idx]
        correct = pred_lbl == labels[idx]
        border  = ('#4CAF50' if correct else
                   '#FF9800' if pred_lbl=='UNCERTAIN' else '#F44336')

        ax_img = axes[0,idx]
        ax_img.imshow(images[idx]); ax_img.axis('off')
        for sp in ax_img.spines.values():
            sp.set_visible(True); sp.set_edgecolor(border); sp.set_linewidth(4)
        ax_img.set_title(
            f"True: {labels[idx]}\nPred: {pred_lbl}\nConf: {conf:.1%}  H={ent:.3f}",
            fontsize=8.5, color=border, fontweight='bold', pad=4)

        ax_bar = axes[1,idx]
        y_pos  = np.arange(len(class_names))
        bars   = ax_bar.barh(y_pos, probs, xerr=stds, color=bar_colors, alpha=0.82,
                             error_kw=dict(ecolor='#555',capsize=4,lw=1.5))
        ax_bar.set_yticks(y_pos); ax_bar.set_yticklabels(class_names, fontsize=8)
        ax_bar.set_xlim(0,1.05); ax_bar.set_xlabel('Probability', fontsize=8)
        ax_bar.axvline(REJECTION_THRESHOLD, color='orange', ls='--',
                       lw=1.2, label=f'Reject<{REJECTION_THRESHOLD}')
        ax_bar.legend(fontsize=6.5, loc='lower right')
        ax_bar.grid(axis='x', alpha=0.3)
        for bar,p in zip(bars,probs):
            ax_bar.text(min(p+0.02,0.98), bar.get_y()+bar.get_height()/2,
                        f'{p:.1%}', va='center', fontsize=7.5, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(MODELS_DIR,f"{dataset_name}_advanced_preds.png"),
                dpi=100, bbox_inches='tight')
    plt.show()
    print(f"✅ Advanced prediction chart saved: {dataset_name}_advanced_preds.png")

# ────────────────────────────────────────────────────────────────────────────
#  5. AUROC + PR-AUC
# ────────────────────────────────────────────────────────────────────────────
def compute_auc_metrics(model, test_gen, class_names, temperature=1.0,
                        tta_steps=TTA_STEPS):
    test_gen.reset()
    images, labels = [], []
    for _ in range(math.ceil(test_gen.samples/test_gen.batch_size)):
        X,y = next(test_gen); images.append(X); labels.append(y)
    images = np.concatenate(images)[:test_gen.samples]
    labels = np.concatenate(labels)[:test_gen.samples]

    aug_gen = ImageDataGenerator(rescale=1.0, horizontal_flip=True,
                                 zoom_range=0.05, rotation_range=5)
    preds = predict_calibrated(model, images, temperature)
    for _ in range(tta_steps-1):
        aug   = np.array([aug_gen.random_transform(im) for im in images])
        preds += predict_calibrated(model, aug, temperature)
    preds /= tta_steps

    print(f"\n{'─'*56}")
    print(f"  📈  AUC Metrics  (TTA={tta_steps}, T={temperature:.2f})")
    print(f"{'─'*56}")
    rows = []
    for i,cls in enumerate(class_names):
        y_bin = labels[:,i]; y_score = preds[:,i]
        if len(np.unique(y_bin))<2:
            print(f"  {cls:<22} N/A (single class)"); continue
        roc = roc_auc_score(y_bin,y_score)
        pr  = average_precision_score(y_bin,y_score)
        print(f"  {cls:<22} AUROC={roc:.4f}   PR-AUC={pr:.4f}")
        rows.append({'Class':cls,'AUROC':roc,'PR-AUC':pr})
    if rows:
        print(f"  {'Macro Average':<22} AUROC={np.mean([r['AUROC'] for r in rows]):.4f}"
              f"   PR-AUC={np.mean([r['PR-AUC'] for r in rows]):.4f}")
    print(f"{'─'*56}")
    return rows

# ────────────────────────────────────────────────────────────────────────────
#  6. CALIBRATION CURVE
# ────────────────────────────────────────────────────────────────────────────
def plot_calibration_curve(model, val_gen, class_names,
                           temperature_before=1.0, temperature_after=None):
    val_gen.reset()
    images, labels = [], []
    for _ in range(math.ceil(val_gen.samples/val_gen.batch_size)):
        X,y = next(val_gen); images.append(X); labels.append(y)
    images = np.concatenate(images)[:val_gen.samples]
    labels = np.concatenate(labels)[:val_gen.samples]
    T_after = temperature_after or temperature_before

    raw_preds = model.predict(images, verbose=0, batch_size=32)
    cal_preds = predict_calibrated(model, images, T_after)

    fig, axes = plt.subplots(1,2,figsize=(12,5))
    fig.suptitle('Calibration Reliability Diagram', fontsize=13, fontweight='bold')

    for ax,preds,title in zip(
            axes, [raw_preds, cal_preds],
            ['Before Temperature Scaling', f'After Temperature Scaling (T={T_after:.2f})']):
        y_true_bin = np.argmax(labels,  axis=-1)
        y_conf     = np.max(preds,      axis=-1)
        y_correct  = (np.argmax(preds, axis=-1)==y_true_bin).astype(float)
        try:
            fp, mpv = calibration_curve(y_correct, y_conf, n_bins=10, strategy='uniform')
        except ValueError:
            fp, mpv = np.array([0,1]), np.array([0,1])
        ece = float(np.mean(np.abs(fp-mpv)))
        ax.plot([0,1],[0,1],'k--',lw=1.2,label='Perfect calibration')
        ax.plot(mpv,fp,'s-',color='#2196F3',lw=2,ms=7,label='Model')
        ax.fill_between(mpv,fp,mpv,alpha=0.15,color='orange',label='Miscalibration gap')
        ax.set_xlabel('Mean Predicted Confidence',fontsize=10)
        ax.set_ylabel('Fraction Correct',fontsize=10)
        ax.set_title(title,fontsize=10); ax.legend(fontsize=9)
        ax.grid(True,alpha=0.3); ax.set_xlim(0,1); ax.set_ylim(0,1)
        ax.text(0.05,0.88,f'ECE={ece:.4f}',transform=ax.transAxes,fontsize=10,
                bbox=dict(boxstyle='round',facecolor='#FFF9C4',alpha=0.8))
    plt.tight_layout()
    plt.savefig(os.path.join(MODELS_DIR,
        f"{class_names[0][:5]}_calibration.png"),dpi=100,bbox_inches='tight')
    plt.show()
    print("✅ Calibration curve saved.")

# ────────────────────────────────────────────────────────────────────────────
#  7. GRAD-CAM++
# ────────────────────────────────────────────────────────────────────────────
def make_gradcampp_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """Grad-CAM++ — better localization via second-order gradients."""
    grad_model = keras.Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as t2:
        with tf.GradientTape() as t1:
            conv_out, preds = grad_model(img_array, training=False)
            if pred_index is None: pred_index = tf.argmax(preds[0])
            score = preds[:, pred_index]
        grads  = t1.gradient(score, conv_out)
    grads2   = t2.gradient(grads, conv_out)
    grads    = tf.cast(grads,    tf.float32)
    grads2   = tf.cast(grads2,   tf.float32)
    conv_out = tf.cast(conv_out, tf.float32)
    denom    = 2.0*grads2 + conv_out*tf.reduce_sum(grads2*grads2,axis=(1,2),keepdims=True)+1e-8
    alpha    = grads2/denom
    weights  = tf.reduce_sum(alpha*tf.nn.relu(grads), axis=(1,2))
    cam      = tf.reduce_sum(weights[:,tf.newaxis,tf.newaxis,:]*conv_out, axis=-1)
    cam      = tf.nn.relu(cam)[0]
    cam      = cam/(tf.reduce_max(cam)+1e-8)
    return cam.numpy()

def visualize_gradcampp(model, dataset_name, class_names, n_images=4):
    """3-row panel: original | CAM++ heatmap | overlay."""
    cfg       = DATASETS[dataset_name]
    test_dir  = os.path.join(cfg['path'],'test')
    img_size  = cfg['img_size']
    last_conv = _find_last_conv(model)
    if not last_conv:
        print("⚠️  No conv layer — skipping Grad-CAM++."); return

    paths, labs = [], []
    for cls in os.listdir(test_dir):
        d = os.path.join(test_dir,cls)
        if not os.path.isdir(d): continue
        imgs = [f for f in os.listdir(d) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        for f in random.sample(imgs, min(n_images//max(len(class_names),1)+1,len(imgs))):
            paths.append(os.path.join(d,f)); labs.append(cls)
    paths,labs = paths[:n_images],labs[:n_images]
    n = len(paths)
    if n==0:
        print("⚠️  No images found."); return

    fig, axes = plt.subplots(3,n,figsize=(4.5*n,13))
    if n==1: axes = axes.reshape(3,1)
    fig.suptitle(f"{dataset_name.upper()} — Grad-CAM++ (Enhanced Localization)",
                 fontsize=13, fontweight='bold')

    for i,(path,label) in enumerate(zip(paths,labs)):
        img = keras.preprocessing.image.load_img(path,target_size=(img_size,img_size))
        arr = keras.preprocessing.image.img_to_array(img)/255.0
        inp = np.expand_dims(arr,0)
        pred    = model.predict(inp,verbose=0)[0]
        p_cls   = class_names[np.argmax(pred)]; conf = np.max(pred)
        try:
            hm      = make_gradcampp_heatmap(inp,model,last_conv)
            hm_rsz  = np.array(keras.preprocessing.image.array_to_img(
                          hm[...,np.newaxis]).resize((img_size,img_size)))/255.0
            cmap    = cm.jet(hm_rsz)[:,:,:3]
            overlay = 0.55*arr+0.45*cmap
        except Exception as e:
            print(f"  CAM++ error: {e}"); cmap=np.zeros((img_size,img_size,3)); overlay=arr
        c = 'green' if p_cls==label else 'red'
        axes[0,i].imshow(arr);    axes[0,i].axis('off')
        axes[0,i].set_title(f"True:{label}\n{p_cls} ({conf:.1%})",fontsize=8.5,color=c,fontweight='bold')
        axes[1,i].imshow(cmap);   axes[1,i].axis('off'); axes[1,i].set_title('CAM++ heatmap',fontsize=8)
        axes[2,i].imshow(overlay);axes[2,i].axis('off'); axes[2,i].set_title('Overlay 55/45',fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(MODELS_DIR,f"{dataset_name}_gradcampp.png"),
                dpi=100,bbox_inches='tight')
    plt.show()
    print(f"✅ Grad-CAM++ saved: {dataset_name}_gradcampp.png")

# ────────────────────────────────────────────────────────────────────────────
#  8. JSON EXPORT  for BI integration
# ────────────────────────────────────────────────────────────────────────────
def export_predictions_json(model, dataset_name, class_names,
                             temperature=1.0, output_dir=MODELS_DIR):
    cfg      = DATASETS[dataset_name]
    test_dir = os.path.join(cfg['path'],'test')
    img_size = cfg['img_size']
    aug_gen  = ImageDataGenerator(rescale=1.0,horizontal_flip=True,
                                  zoom_range=0.05,rotation_range=5)
    records = []
    for cls in class_names:
        d = os.path.join(test_dir,cls)
        if not os.path.isdir(d): continue
        files = [f for f in os.listdir(d) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        for fname in files:
            img = keras.preprocessing.image.load_img(
                os.path.join(d,fname),target_size=(img_size,img_size))
            arr = keras.preprocessing.image.img_to_array(img)/255.0
            inp = np.expand_dims(arr,0)
            preds = predict_calibrated(model,inp,temperature)
            for _ in range(TTA_STEPS-1):
                aug   = np.array([aug_gen.random_transform(arr)])
                preds += predict_calibrated(model,aug,temperature)
            preds /= TTA_STEPS
            pc   = class_names[int(np.argmax(preds[0]))]
            conf = float(np.max(preds[0]))
            ent  = float(-np.sum(preds[0]*np.log(preds[0]+1e-9)))
            records.append({
                'file':fname,'true_class':cls,'pred_class':pc,
                'correct':cls==pc,'confidence':round(conf,4),'entropy':round(ent,4),
                'status':'uncertain' if conf<REJECTION_THRESHOLD else 'classified',
                'class_probs':{c:round(float(p),4) for c,p in zip(class_names,preds[0])}
            })
    ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = os.path.join(output_dir,f"{dataset_name}_predictions_{ts}.json")
    with open(path,'w') as f:
        _json.dump({'dataset':dataset_name,'temperature':temperature,
                    'rejection_threshold':REJECTION_THRESHOLD,'predictions':records},f,indent=2)
    n_ok  = sum(1 for r in records if r['correct'])
    n_rej = sum(1 for r in records if r['status']=='uncertain')
    print(f"  ✅ Exported {len(records)} records → {path}")
    print(f"     Correct: {n_ok}/{len(records)} ({n_ok/max(len(records),1):.1%})"
          f"  |  Rejected: {n_rej} ({n_rej/max(len(records),1):.1%})")
    return path

print("✅ Advanced prediction pipeline v3.0 defined:")
print("   1. Temperature Scaling    5. AUROC / PR-AUC")
print("   2. MC-Dropout uncertainty 6. Calibration Curve")
print("   3. Ensemble TTA+MC        7. Grad-CAM++")
print("   4. Rejection threshold    8. JSON Export")


## ▶️ Run Advanced Predictions — All Models

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  Run advanced prediction pipeline on all three models
# ═══════════════════════════════════════════════════════════════════════════
models_info = [
    (cabin_model,   'cabin_cleanliness', cabin_classes),
    (crowd_model,   'crowd',             crowd_classes),
    (luggage_model, 'luggage',           luggage_classes),
]
all_temperatures = {}

for model, dname, cnames in models_info:
    print(f"\n{'═'*72}")
    print(f"  🔬  ADVANCED PREDICTIONS: {dname.upper()}")
    print(f"{'═'*72}")

    _, val_gen_t, test_gen_t = load_data(dname)

    # 1. Temperature scaling
    print("\n📐 Step 1 — Fitting temperature on validation set…")
    T = find_temperature(model, val_gen_t)
    all_temperatures[dname] = T

    # 2. AUROC / PR-AUC
    print(f"\n📈 Step 2 — AUC metrics (T={T:.2f})…")
    compute_auc_metrics(model, test_gen_t, cnames, temperature=T)

    # 3. Calibration curve
    print("\n📊 Step 3 — Calibration reliability diagram…")
    _, val_gen_cal, _ = load_data(dname)
    plot_calibration_curve(model, val_gen_cal, cnames,
                           temperature_before=1.0, temperature_after=T)

    # 4. Advanced prediction display (images + prob bars + MC uncertainty)
    print("\n🖼️  Step 4 — Ensemble prediction display…")
    visualize_predictions_advanced(model, dname, cnames,
        num_samples=6, temperature=T, mc_passes=15, tta_steps=TTA_STEPS)

    # 5. Grad-CAM++
    print("\n🔥 Step 5 — Grad-CAM++ visualisation…")
    visualize_gradcampp(model, dname, cnames, n_images=4)

    # 6. JSON export
    print("\n💾 Step 6 — Exporting predictions to JSON…")
    export_predictions_json(model, dname, cnames, temperature=T)

print("\n"+"═"*72)
print("  🏁  ALL ADVANCED PREDICTIONS COMPLETE")
print("  Temperatures:", {k:f"{v:.2f}" for k,v in all_temperatures.items()})
print("═"*72)


## 📊 Enhanced Final Summary (Calibrated)

In [ ]:
summary_rows = []
for model, dname, cnames in models_info:
    _, _, test_gen_s = load_data(dname)
    T = all_temperatures.get(dname, 1.0)
    test_gen_s.reset()
    imgs, lbls = [], []
    for _ in range(math.ceil(test_gen_s.samples/test_gen_s.batch_size)):
        X,y = next(test_gen_s); imgs.append(X); lbls.append(y)
    imgs = np.concatenate(imgs)[:test_gen_s.samples]
    lbls = np.concatenate(lbls)[:test_gen_s.samples]
    y_true = np.argmax(lbls, axis=-1)

    aug_g = ImageDataGenerator(rescale=1.0, horizontal_flip=True,
                                zoom_range=0.05, rotation_range=5)
    preds = predict_calibrated(model, imgs, T)
    for _ in range(TTA_STEPS-1):
        aug   = np.array([aug_g.random_transform(im) for im in imgs])
        preds += predict_calibrated(model, aug, T)
    preds /= TTA_STEPS
    y_pred = np.argmax(preds, axis=-1)
    acc_cal = float(accuracy_score(y_true,y_pred))
    f1_cal  = float(f1_score(y_true,y_pred,average='weighted',zero_division=0))
    n_rej   = int(np.sum(np.max(preds,axis=-1)<REJECTION_THRESHOLD))
    summary_rows.append({
        'Model'             : dname.upper(),
        'Backbone'          : DATASETS[dname]['backbone'],
        'Acc (calibrated)'  : f"{acc_cal:.4f}",
        'F1-w (calibrated)' : f"{f1_cal:.4f}",
        'Temperature'       : f"{T:.2f}",
        f'Rejected (<{REJECTION_THRESHOLD})': f"{n_rej} ({n_rej/max(len(preds),1):.1%})"
    })

summary_df = pd.DataFrame(summary_rows)
print("\n"+"═"*95)
print("  🏆  ENHANCED FINAL SUMMARY  (Temperature-Calibrated + Rejection)")
print("═"*95)
print(summary_df.to_string(index=False))
print("═"*95)

fig, ax = plt.subplots(figsize=(11,5))
x = np.arange(len(summary_rows)); w = 0.30
b1 = ax.bar(x-w/2,[float(r['Acc (calibrated)'])  for r in summary_rows],w,
            label='Acc (Calibrated)',  color='#2196F3',alpha=0.85)
b2 = ax.bar(x+w/2,[float(r['F1-w (calibrated)']) for r in summary_rows],w,
            label='F1-w (Calibrated)', color='#4CAF50',alpha=0.85)
ax.bar_label(b1,fmt='%.3f',fontsize=9,padding=3)
ax.bar_label(b2,fmt='%.3f',fontsize=9,padding=3)
ax.set_xticks(x); ax.set_xticklabels([r['Model'] for r in summary_rows],fontsize=11)
ax.set_ylim(0,1.12); ax.set_ylabel('Score',fontsize=11)
ax.set_title('SkyInsight — Temperature-Calibrated Model Comparison',
             fontsize=13,fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y',alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR,'enhanced_comparison_chart.png'),dpi=120)
plt.show()
print("✅ Enhanced comparison chart saved.")


## ✅ Complete!

### Training enhancements:
| Enhancement | Typical gain |
|---|---|
| EfficientNetV2S vs B0 | +2–4% accuracy |
| GeM pooling vs GAP | +0.5–1.5% |
| AdamW + weight decay | Reduces overfitting |
| Cosine annealing + warmup | Smoother convergence |
| Mixup augmentation | +0.5–2% generalization |
| TTA (×5) | +0.5–2% test accuracy |

### v3.0 prediction enhancements:
| Enhancement | Benefit |
|---|---|
| Temperature Scaling | Calibrated, honest confidence |
| MC-Dropout (×15) | Uncertainty quantification |
| Ensemble TTA+MC | Maximum robustness |
| Rejection threshold | No silent misclassification |
| Grad-CAM++ | Sharper visual explanations |
| AUROC / PR-AUC | Richer per-class evaluation |
| Calibration curve | Diagnosable confidence quality |
| JSON export | BI dashboard integration |
